# 99 · Final analysis: development-CV report, freeze, descriptive holdout, final tables

1. **Development-CV report** of every completed result and the category C participants (difficult to classify
   across the development-CV models; reported with their QC flags, never excluded).
2. **Freeze** the final analysis (`configs/final/holdout.yaml`): every pre-specified variant of
   exp0, exp1, exp2, exp3, exp4, exp5, exp6, exp8, window mode B, lcmv and wmne, on the recorded atlas
   and graph construction. It refuses while any of these units lacks a completed development-CV result or a decision
   is missing, and it is written once (`FINAL_v1/decisions/final_analysis.json`, with the code commit).
   Exp 7 (exploratory), mode A, the atlas not chosen and the graph constructions not selected stay development-only.
3. **Holdout**: each frozen unit is fitted on the 52 development participants and evaluated once on the 13 holdout
   participants. The code refuses the holdout for any unit outside the frozen analysis. Descriptive only.
4. **Final tables**: all results, the matched LCMV vs wMNE and mode B vs A comparisons (paired DeLong, descriptive).

**Holdout policy.** The 13-participant holdout was inspected during earlier experiments (LX / WX suites), so it is no longer an untouched test set. In this release it is evaluated once per final configuration and reported descriptively; it is never used to select architectures, graph constructions, k, atlas resolution, QC thresholds or any other setting (selection reads development CV results only).

GPU runtime (T4) for step 3; resumable.

**How to use:** *Runtime → Run all*. Allow Google Drive access when asked. If Colab disconnects, open the notebook
again and *Run all* again: finished work is kept on Google Drive and skipped. Everything is written to
`<outputs>/FINAL_v1/` on Drive (never to the temporary Colab disk); the validated graph stores and all earlier
results are only read.

In [ ]:
# ===== Setup: nothing to edit. Mounts Drive, gets the code, installs the pinned environment. =====
UPDATE_CODE = False   # leave False. True only to move the release code pin deliberately to the newest commit
                      # (<outputs>/FINAL_v1/CODE_COMMIT.txt; unfinished work is then recomputed).
BRANCH = 'claude/unified-final-framework'
REPO_URL = 'https://github.com/Samimj9/Source-space-EEG-graph-learning-for-Alzheimer-s-disease-using-multi-band-functional-connectivity.git'

import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

def _run(cmd, cwd=None):  # shows output and stops "Run all" on failure
    p = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if p.returncode:
        raise RuntimeError(f'failed: {" ".join(cmd)}')

os.chdir('/content')
_run(['rm', '-rf', '/content/repo'])
_run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'])
os.chdir('/content/repo')
_run([sys.executable, 'scripts/colab_setup.py'] + (['--update-code'] if UPDATE_CODE else []))
sys.path.insert(0, '/content/repo/src')   # the package, for this notebook's own cells

from adgraph.colab import sh
from adgraph.config import load_paths
PATHS = load_paths('configs/paths.yaml')
OUT = PATHS['processed_dir']
FINAL = OUT + '/FINAL_v1'
print('release outputs on Drive:', FINAL)

## 1. Development-CV report and difficult participants

In [ ]:
sh("python scripts/final.py report")
sh("python scripts/final.py difficult")

## 2. Freeze the final analysis (once)

In [ ]:
sh("python scripts/final.py freeze")

## 3. Holdout of the frozen units only (descriptive)

In [ ]:
ERRORS = []
for exp in ["exp0", "exp1", "exp2", "exp3", "exp4", "exp5", "exp6", "exp8"]:
    try:
        sh(f"python scripts/final.py holdout --exp {exp}")
    except RuntimeError:
        ERRORS.append(exp)
if ERRORS:
    raise RuntimeError(f'holdout units did not complete for {ERRORS}; completed units are saved. Run all again.')

## 4. Final tables (holdout rows are descriptive only)

In [ ]:
import glob, json
import pandas as pd
from IPython.display import Image, display
pd.set_option('display.max_rows', 400, 'display.width', 250, 'display.max_columns', 40)

def latest(pattern):
    found = sorted(glob.glob(pattern))
    return found[-1] if found else None
sh("python scripts/final.py report")
d = latest(FINAL + '/reports/tables_*')
for f in ('all_results.csv', 'lcmv_vs_wmne.csv', 'mode_B_vs_A.csv'):
    if os.path.exists(f'{d}/{f}'):
        print('====', f)
        display(pd.read_csv(f'{d}/{f}').round(3))